In [8]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
 
# --- Paths ---
BASE_DIR      = os.path.abspath("..")
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed", "pH")
SHAPES_DIR    = os.path.join(BASE_DIR, "data", "raw", "shapes")
CMIP6_DIR     = os.path.join(BASE_DIR, "data", "raw", "CMIP6")
FIGURES_DIR   = os.path.join(BASE_DIR, "figures")

In [9]:
ds_BBI  = xr.open_dataset(os.path.join(PROCESSED_DIR, "ph_BBI_ready.nc"))
ds_BBII = xr.open_dataset(os.path.join(PROCESSED_DIR, "ph_BBII_ready.nc"))

def weighted_mean(ds):
    weights = np.cos(np.deg2rad(ds["latitude"]))
    return ds["ph"].weighted(weights).mean(
        dim=["latitude","longitude"], skipna=True)

ph_mean_BBI  = weighted_mean(ds_BBI)
ph_mean_BBII = weighted_mean(ds_BBII)

# Promedio de ambas zonas
ph_combined = (ph_mean_BBI + ph_mean_BBII) / 2
df_obs = ph_combined.to_dataframe(name="pH").reset_index()
df_obs["time"] = pd.to_datetime(df_obs["time"])

ph_obs_2020 = df_obs[df_obs["time"].dt.year == 2020]["pH"].mean()
print(f"pH BB 2020 CMEMS: {ph_obs_2020:.4f}")

pH BB 2020 CMEMS: 8.0514


In [10]:
ds_245 = xr.open_dataset(os.path.join(CMIP6_DIR, "pHT_median_ssp245.nc"))

lat2d = ds_245["latitude"].values
lon2d = ds_245["longitude"].values
ph2d  = ds_245["pHT"].values  # (9, 180, 360)

# Máscara del BB
mask_bb = (
    (lat2d >= -56) & (lat2d <= -52) &
    (lon2d >= -64) & (lon2d <= -54)
)

ph_bb_2020_cmip6 = ph2d[0][mask_bb].mean()
print(f"pH BB 2020 CMIP6 SSP2-4.5: {ph_bb_2020_cmip6:.4f}")
print(f"pH BB 2020 CMEMS          : {ph_obs_2020:.4f}")
print(f"Bias (CMIP6 - CMEMS)      : {ph_bb_2020_cmip6 - ph_obs_2020:.4f}")

pH BB 2020 CMIP6 SSP2-4.5: nan
pH BB 2020 CMEMS          : 8.0514
Bias (CMIP6 - CMEMS)      : nan


C:\Users\gisel\AppData\Local\Temp\ipykernel_86280\4255156097.py:13: RuntimeWarning: Mean of empty slice.
  ph_bb_2020_cmip6 = ph2d[0][mask_bb].mean()
C:\Users\gisel\anaconda3\envs\BB_stress_paper\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [11]:
print(f"lat rango: {lat2d.min():.2f} → {lat2d.max():.2f}")
print(f"lon rango: {lon2d.min():.2f} → {lon2d.max():.2f}")

lat rango: -89.50 → 89.50
lon rango: 20.50 → 379.50


In [12]:
# Convertir longitudes del BB a convención 0-360
lon_min = -64 + 360  # = 296
lon_max = -54 + 360  # = 306

mask_bb = (
    (lat2d >= -56) & (lat2d <= -52) &
    (lon2d >= lon_min) & (lon2d <= lon_max)
)

print(f"Píxeles en el BB: {mask_bb.sum()}")
ph_bb_2020_cmip6 = ph2d[0][mask_bb].mean()
print(f"pH BB 2020 CMIP6 SSP2-4.5: {ph_bb_2020_cmip6:.4f}")
print(f"pH BB 2020 CMEMS          : {ph_obs_2020:.4f}")
print(f"Bias (CMIP6 - CMEMS)      : {ph_bb_2020_cmip6 - ph_obs_2020:.4f}")

Píxeles en el BB: 40
pH BB 2020 CMIP6 SSP2-4.5: nan
pH BB 2020 CMEMS          : 8.0514
Bias (CMIP6 - CMEMS)      : nan


In [14]:
# Ver cuántos NaN hay en esos píxeles
ph_bb_vals = ph2d[0][mask_bb]
print(f"Valores en el BB: {ph_bb_vals}")
print(f"NaN: {np.isnan(ph_bb_vals).sum()} de {len(ph_bb_vals)}")

# Ver todos los timesteps
for i, year in enumerate([2020, 2030, 2040, 2050, 2060, 2070, 2080, 2090, 2100]):
    vals  = ph2d[i][mask_bb]
    valid = vals[~np.isnan(vals)]
    mean  = f"{valid.mean():.4f}" if len(valid) > 0 else "NaN"
    print(f"{year}: {len(valid)} válidos, media={mean}")

Valores en el BB: [8.059 8.059 8.058 8.057 8.055 8.055 8.057 8.06  8.062 8.065 8.069 8.066
 8.062 8.057 8.053 8.051 8.053 8.057 8.06  8.062 8.077 8.074 8.07  8.063
 8.047 8.044 8.047 8.052 8.055 8.057 8.081 8.078 8.078 8.081   nan   nan
 8.045 8.05  8.052 8.054]
NaN: 2 de 40
2020: 38 válidos, media=8.0601
2030: 38 válidos, media=8.0342
2040: 38 válidos, media=8.0079
2050: 38 válidos, media=7.9817
2060: 38 válidos, media=7.9578
2070: 38 válidos, media=7.9403
2080: 38 válidos, media=7.9209
2090: 38 válidos, media=7.9142
2100: 38 válidos, media=7.9093
